# 11 . The bail-out rule as the ONLY exit

Same **entry** and same **stop** as notebook 10 — but the take-profit is removed entirely and the
*secondary* bail-out rule becomes the sole way a trade closes in profit.

> **entry** `support_break_short` · **stop** `leg_high` · **TP** `none@v1` · **exit rule** = a secondary

So a trade can end only three ways: the **bail-out** (in profit), the **stop-loss**, or forced close at
the end of data.

### Two flavours of the bail-out

| Rule | Arms when | Use |
|---|---|---|
| `shaky_trend_bail@v1` | price closes above / **touches the EMA line** after entry | the original: only bail once the trend looks shaky |
| `profit_candle_bail@v1` | **immediately** (always armed) | "ride it while candles go my way, take what's there on the first one against me" |

Both keep the **in-profit gate** (enforced in the engine): the rule can *never* close a losing trade —
a loser is always handed to the stop. That asymmetry is the whole point of section 5, which is the
most important cell in this notebook.

## 0 . Setup

In [ ]:
import sys, os, warnings; warnings.filterwarnings('ignore')
try:
    import quant
except ModuleNotFoundError:
    sys.path.insert(0, os.path.abspath('..')); import quant
import numpy as np, pandas as pd
from quant.data import get_ohlcv
from quant import scenarios as sc
from quant.scenarios import Preset
from quant.viz import scenario_figure, zoom_to_trade, open_in_browser, CROSSHAIR_JS
pd.set_option('display.max_rows', 200); pd.set_option('display.width', 220)
print('secondaries:', sorted(sc.SECONDARIES))

## 1 . Data
A single day gives 1–2 trades — far too few to judge anything. Widen `START`/`END` for a real read.

In [ ]:
SYMBOL, SOURCE, MARKET, TF = 'BTCUSDT', 'binance', 'spot', '1m'
START, END = '2026-07-10', '2026-07-29'
df = get_ohlcv(SYMBOL, TF, start=START, end=END, source=SOURCE, market=MARKET, tz='UTC')
print(f'{len(df):,} {TF} bars   {df.t.min()}  ->  {df.t.max()}')

## 2 . Config — entry & stop unchanged; the bail-out is the exit

In [ ]:
SCENARIO  = 'support_break_short@v1.2'   # entry: unchanged from notebook 10
STOP      = 'leg_high@v1'                # stop:  unchanged from notebook 10
TP        = 'none@v1'                    # NO take-profit — the bail-out is the only way to win
EXIT_RULE = 'profit_candle_bail@v1'      # 'profit_candle_bail@v1' (always armed)
                                         # 'shaky_trend_bail@v1'   (arms on an EMA touch first)

PARAMS = dict(
    # -- entry (same as nb 10) --
    entry_mode='break_close', break_mode='close_below', prev_low_n=20, confirm_red=2,
    grad_period=200, grad_smooth=9, grad_lookback=5, grad_max=0.0,
    # -- stop (same as nb 10) --
    stop_buffer_pct=0.05, stop_max_risk_pct=0.0, stop_pct=0.6, stop_above_ema_only=False,
    # -- THE EXIT RULE --
    sec_opp_n=1,          # close after N opposite-colour (green) candles, while in profit
    candle_src='real',    # 'real' or 'ha'  -> which candles the colour test reads
    sec_touch=True,       # shaky_trend_bail only: a TOUCH of the EMA arms it (not just a close above)
)

# allow_leverage=True so risk_pct_equity can actually reach 1% — otherwise a tight stop on a
# 60k instrument caps the position at 100% of cash and the real risk is far below 1% (see nb 10).
CFG = dict(initial_cash=10_000, spread=0.0, sizing_mode='risk_pct_equity', sizing_value=1.0,
           allow_leverage=True)

run = sc.run_pieces(df, SCENARIO, STOP, TP, secondary=EXIT_RULE, params=PARAMS, cfg=CFG)
print(sc.resolve_secondary(EXIT_RULE).description)
s = run.stats
print(f"\ntrades {int(s['num_trades'])} | win {s['win_rate_pct']:.0f}% | ret {s['total_return_pct']:+.2f}% "
      f"| dd {s['max_drawdown_pct']:.2f}% | PF {s['profit_factor']:.2f} | Sharpe {s['sharpe']:.2f}")
print('close reasons:', dict(run.trades.close_reason.value_counts()) if len(run.trades) else {})

## 3 . Trades

In [ ]:
run.trades[['side','entry_time','exit_time','entry_price','stop_price','exit_price',
            'pnl','bars_held','close_reason']]

## 4 . ⚠ The asymmetry — read this before trusting the win rate
The bail-out can only ever close a **winner** (the in-profit gate), so every loss is a full stop-out.
That mechanically produces a **high win rate**, which tells you nothing on its own. What matters is
whether the average win is big enough to pay for a −1R loss.

Everything below is in **R** (multiples of the risk taken on that trade = |stop − entry| × qty), which
removes position sizing from the comparison.

> **Expectancy = win% × avgWin_R − loss% × avgLoss_R.** It must be **> 0**. A 75% win rate with
> +0.2R wins and −1R losses loses money: 0.75(0.2) − 0.25(1.0) = **−0.10R** per trade.

In [ ]:
def r_table(r):
    t = r.trades.copy()
    if not len(t):
        return pd.DataFrame()
    t['risk'] = (t.stop_price - t.entry_price).abs() * t.qty
    t['R'] = t.pnl / t.risk
    return t

def expectancy(r, tag=''):
    t = r_table(r)
    if not len(t):
        return dict(config=tag, trades=0)
    wins, losses = t[t.pnl > 0], t[t.pnl <= 0]
    wr = len(wins) / len(t)
    aw = wins.R.mean() if len(wins) else 0.0
    al = abs(losses.R.mean()) if len(losses) else 0.0
    return dict(config=tag, trades=len(t), win_pct=round(wr * 100, 1),
                avg_win_R=round(aw, 3), avg_loss_R=round(-al, 3),
                expectancy_R=round(wr * aw - (1 - wr) * al, 4),
                total_R=round(t.R.sum(), 2), ret_pct=round(r.stats['total_return_pct'], 2))

print('Per-trade R multiples:')
display(r_table(run)[['entry_time','pnl','R','close_reason','bars_held']])
print('\nExpectancy:')
pd.DataFrame([expectancy(run, EXIT_RULE)])

## 5 . Is the bail-out better than a plain target?
The same entry and stop, exited five different ways. Judge on **expectancy_R**, not win%.

In [ ]:
variants = [
    ('rr 1R target',        'rr@v1',      None,                     dict(tp_rr=1.0)),
    ('rr 2R target',        'rr@v1',      None,                     dict(tp_rr=2.0)),
    ('trailing 1%',         'trailing@v1', None,                    dict(trail_pct=1.0)),
    ('bail: shaky (armed)', 'none@v1',    'shaky_trend_bail@v1',    {}),
    ('bail: profit candle', 'none@v1',    'profit_candle_bail@v1',  {}),
    ('2R + shaky as backup','rr@v1',      'shaky_trend_bail@v1',    dict(tp_rr=2.0)),
]
rows = []
for tag, tp, sec, over in variants:
    r = sc.run_pieces(df, SCENARIO, STOP, tp, secondary=sec, params={**PARAMS, **over}, cfg=CFG)
    row = expectancy(r, tag)
    row['reasons'] = dict(r.trades.close_reason.value_counts()) if len(r.trades) else {}
    rows.append(row)
pd.DataFrame(rows)

## 6 . Tuning the bail-out
`sec_opp_n` = how many candles must close against you before bailing (patience), and whether the
colour test reads **real** or **Heikin-Ashi** candles (HA is smoother, so it bails less often).

In [ ]:
rows = []
for src in ['real', 'ha']:
    for n in [1, 2, 3]:
        r = sc.run_pieces(df, SCENARIO, STOP, TP, secondary=EXIT_RULE,
                          params={**PARAMS, 'sec_opp_n': n, 'candle_src': src}, cfg=CFG)
        rows.append(expectancy(r, f'{src}, sec_opp_n={n}'))
pd.DataFrame(rows)

## 7 . Chart
Purple ▲ = the bar the rule armed; purple ✡ = the bail-out exit. Amber dashed = level B.

In [ ]:
fig = scenario_figure(run)
print('chart ->', open_in_browser(fig, 'bail_exit_chart.html', extra_js=CROSSHAIR_JS))

In [ ]:
if len(run.trades):
    fig1 = scenario_figure(run)
    zoom_to_trade(fig1, run, run.trades.index[0], pad_bars=60)
    print('zoom ->', open_in_browser(fig1, 'bail_exit_trade.html', extra_js=CROSSHAIR_JS))

## 8 . Save as a preset

In [ ]:
preset = Preset(name='btc_bail_exit_only', scenario=SCENARIO, stop=STOP, tp=TP, secondary=EXIT_RULE,
                params=PARAMS, cfg=CFG,
                data=dict(symbol=SYMBOL, interval=TF, start=START, end=END, source=SOURCE,
                          market=MARKET, tz='UTC'),
                description='Support-break short exited solely by the in-profit bail-out rule.')
print('saved ->', preset.to_json('../scenarios/presets/btc_bail_exit_only.json'))

## 9 . How to read this
- **A high win rate is expected here and means nothing by itself.** The bail-out is gated on being in
  profit, so it can only ever book winners; every loser goes to the stop. Win rate is an artefact of
  the rule's construction, not evidence of edge.
- **`expectancy_R` is the number that decides it.** It must be positive. Wins capped at "first candle
  against me" tend to be small (well under +1R) while losses are a full −1R, so this style needs a
  *very* high win rate to break even.
- **Levers if expectancy is negative:** raise `sec_opp_n` (more patience, bigger wins), use
  `candle_src='ha'` (smoother, bails less), tighten the stop so −1R is a smaller move, or keep a
  target and use the bail-out only as a backup (the last row of section 5).
- **Sample size.** One day gives 1–2 trades. Widen `START`/`END` in section 1 and re-run before
  drawing any conclusion; everything here is in-sample on one symbol.